# Projekt MSP1
Cílem tohoto projektu je se seznámit s programovými nástroji využívaných ve statistice a osvojit si základní procedury. Projekt není primárně zaměřen na efektivitu využívání programového vybavení (i když úplně nevhodné konstrukce mohou mít vliv na hodnocení), ale nejvíce nás zajímají vaše statistické závěry a způsob vyhodnocení. Dbejte také na to, že každý graf musí splňovat nějaké podmínky - přehlednost, čitelnost, popisky.

V projektu budete analyzovat časy běhu šesti různých konfigurací algoritmů. Ke každé konfiguraci vzniklo celkem 200 nezávislých běhů, jejichž logy máte k dispozici v souboru [logfiles.zip](logfiles.zip).

Pokud nemáte rozchozené prostředí pro pro spouštění Jupyter notebooku, můžete využití službu [Google Colab](https://colab.google/). Jakákoliv spolupráce, sdílení řešení a podobně je zakázána!

S případnými dotazy se obracejte na Vojtěcha Mrázka (mrazek@fit.vutbr.cz).

__Odevzdání:__ tento soubor (není potřeba aby obsahoval výstupy skriptů) do neděle 22. 10. 2023 v IS VUT. Kontrola bude probíhat na Pythonu 3.10.12; neočekává se však to, že byste používali nějaké speciality a nekompatibilní knihovny. V případě nesouladu verzí a podobných problémů budete mít možnost reklamace a prokázání správnosti funkce. Bez vyplnění vašich komentářů a závěrů do označených buněk nebude projekt hodnocen!

__Upozornění:__ nepřidávejte do notebooku další buňky, odpovídejte tam, kam se ptáme (textové komentáře do Markdown buněk)

__Tip:__ před odevzdáním resetujte celý notebook a zkuste jej spustit od začátku. Zamezíte tak chybám krokování a editací, kdy výsledek z buňky na konci použijete na začátku.

__OTÁZKA K DOPLNĚNÍ:__

Matúš Remeň, xremen01

## Načtení potřebných knihoven
Načtěte knihovny, které jsou nutné pro zpracování souborů a práci se statistickými funkcemi. Není dovoleno načítat jiné knihovny.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns
from zipfile import ZipFile

## Načtení dat do DataFrame
Ze souboru `logfiles.zip` umístěném ve stejném adresáři načtěte data a vytvořte Pandas DataFrame.

Z logu vás budou nejvíce zajímat řádky
```
Configuration: config6
Run: 191
Time of run: 53.298725254089774
```

Můžete využít následující kostru - je vhodné pracovat přímo se ZIP souborem. Jedinou nevýhodou je to, že vám bude vracet _byte_ objekt, který musíte přes funkci `decode` zpracovat.

In [ ]:
# TODO nacteni dat ze zip souboru

def load_logfile(f) -> dict:
    """Load a logfile from a file-like object and return a dict with the data."""
    data = {
        "conf": None,
        "run": None,
        "time": np.nan
    }

    for line in f:
        line = line.decode("utf-8")

        # MR: naplnenie slovniku datami zo suboru
        if line.startswith("Configuration:"):
            data["conf"] = line.split(":")[1].strip()
        elif line.startswith("Run:"):
            data["run"] = line.split(":")[1].strip()
        elif line.startswith("Time of run:"):
            data["time"] = float(line.split(":")[1].strip())
        
    return data

data = []
with ZipFile("logfiles.zip") as zf:
    for filename in zf.namelist():
        with zf.open(filename, "r") as f:
            data.append(load_logfile(f))
df = pd.DataFrame(data)
df

## Analýza a čištění dat
Vhodným způsobem pro všechny konfigurace analyzujte časy běhů a pokud tam jsou, identifikujte hodnoty, které jsou chybné. 

In [ ]:
# TODO vykresleni grafu pro identifikace outlieru
def get_config_data() -> list:
    configs = []
    for i in range(1, 7):
        config = np.array(df[df["conf"] == f"config{i}"]["time"], dtype=float)
        configs.append(config)
    return configs

fig, ax = plt.subplots(figsize=(16, 9))
ax.boxplot(get_config_data())
ax.set_xticks([1, 2, 3, 4, 5, 6])
ax.set(title="Run times per configuration (raw)", xlabel="Configurations", ylabel="Run times [s]", ylim=(0, None))

__OTÁZKA K DOPLNĚNÍ:__

_Objevily se nějaké chybné hodnoty? Proč tam jsou s ohledem na to, že se jedná o běhy algoritmů?_ </br>
Áno. Sú to hodnoty extrémov v jednotlivých konfiguráciách. S ohladom, že sa jendá o dobu behu algoritmov, outliers s maximálnymi hodnotami z datasetu môžu predstavovať beh, v ktorom sa algoritmus zasekol/zacyklil. Outliers s minimálnymi hodnotami z datasetu môžu predstavovať neočakávané chyby na začiatku algoritmu, ktoré viedli k skoršiemu ukončeniu behu alebo sa algoritmus ani nepodarilo spustit.

Vyčistěte dataframe `df` tak, aby tam tyto hodnoty nebyly a ukažte znovu analýzu toho, že čištění dat bylo úspěšné. Odtud dále pracujte s vyčištěným datasetem.

In [ ]:
# TODO kod pro upravu dataframe tak, že tam tyto hodnoty nebudou:
df = df[(1 < df["time"]) & (df["time"] < 3500)]

## Deskriptivní popis hodnot
Vypište pro jednotlivé konfigurace základní deskriptivní parametry času pro jednotlivé konfigurace.  

__TIP__ pokud výsledky uložíte jako Pandas DataFrame, zobrazí se v tabulce.

In [ ]:
# TODO deskriptivni parametry
desc_params = []
for i in range(1, 7):
    config = np.array(df[df["conf"] == f"config{i}"]["time"], dtype=float)
    desc_params.append({
        "name": f"config{i}",
        "mean": config.mean(),
        "median": np.median(config),
        "std": config.std(),
        "variance": config.var(),
        "min": config.min(),
        "p25": np.percentile(config, 25),
        "p50": np.percentile(config, 50),
        "p75": np.percentile(config, 75),
        "max": config.max(),
    })
desc_params = pd.DataFrame(desc_params)
desc_params

__OTÁZKA K DOPLNĚNÍ:__

_Okomentujte, co všechno můžeme z parametrů vyčíst._ </br>
Z parametov je vidno, že 1. konfigurácia v priemere doshovala najlepšie výsledky, čo sa týka doby behu algoritmu - najnižší priemer a taktiež rozptyl hodnôt. Konfigurácia 4 síce dosiahla najnižšiu minimálnu dobu behu algoritmu, ale kvoli velkému rozptylu hodnôt algoritmus s touto konfiguráciou nemá veľmi stabilnú dobu behu (veľké výkyvy doby behy algoritmu pre rôzne vstupy). Na základe viacerých parametrov, konfigurácia 5 dosahovala najhoršie výsledky - vysoký rozptyl, vysoká priemerná doba behu, najvyššia min a max doba behu.

## Vizualizace
Vizualizujte časy běhů algoritmů v jednom kompaktním grafu tak, aby byl zřejmý i rozptyl hodnot. Zvolte vhodný graf, který pak níže komentujte.

In [ ]:
# TODO graf
configs = get_config_data()
fig, ax = plt.subplots(figsize=(16, 9))
ax.boxplot(configs)
ax.set_xticks([1, 2, 3, 4, 5, 6])
ax.set(title="Run times per configuration (filtered exceptional outliers)", xlabel="Configurations", ylabel="Run times [s]", ylim=(0, None))

means = [np.mean(conf) for conf in configs]
ax.scatter(range(1, 7), means, color="red", marker=".", label="Mean")
ax.legend()

__OTÁZKA K DOPLNĚNÍ:__

_Okomentujte  výsledky z tabulky._</br>
Na grafe typu Box Plot vyššie, sú vizualizované výsledky jednotlivých konfigurácií. Zobrazené sú priemery, mediány, percentily 25 a 75, minimá a maximá.

## Určení efektivity konfigurací algoritmů
Nás ale zajímá, jaká konfigurace je nejrychlejší. Z výše vykresleného grafu můžeme vyloučit některé konfigurace. Existuje tam však minimálně jedna dvojice, u které nedokážeme jednoznačně určit, která je lepší - pokud nebudeme porovnávat pouze extrémní hodnoty, které mohou být dané náhodou, ale celkově. Proto proveďte vhodný test významnosti - v následující části diskutujte zejména rozložení dat (i s odkazem na předchozí buňky, variabilitu vs polohu a podobně). Je nutné každý logický krok a výběry statistických funkcí komentovat. Můžete i přidat další buňky.

Vužijte vhodnou funkci z knihovny `scipy.stats` a funkci poté __implementujte sami__ na základě základních matematických funkcí knihovny `numpy` případně i funkcí pro výpočet studentova rozložení v [scipy.stats](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.t.html). Při vlastní implementaci není nutné se primárně soustředit na efektivitu výpočtu (není potřeba využít všechny funkce numpy, můžete použít normální cykly a podobně - v hodnocení však bude zahrnuta přehlednost a neměly by se objevit jasné chyby, jako je zvýšení třídy složitosti a podobně).

__OTÁZKA K DOPLNĚNÍ:__

_Jaká data budete zkoumat? Jaké mají rozložení a parametry (např. varianci) a jaký test použijete? Jaká je nulová hypotéza? Jak se liší variabilita a poloha vybraných konfigurací?_</br>

Budem skúmať dáta Config1 a Config4, lebo Config1 dosiahla najlepší priemerný čas, ale Config4 minimálny čas - sú teda všeobecne rovnako rýchle, alebo nie.
Použijem Welch’s t-test na nulovú hypotézu, lebo porovnávam dve nezávislé vzorky s líšiacimi sa rozptylmi.</br>
**Nulová hypotéza**: Config1 a Config4 sú rovnaké.</br>
**Alternatívna hypotéza**: Config1 je lepšia ako Config4, alebo Config4 je lepšia ako Config1.</br>
Config1 má menší rozptyl hodnôt, lepší priemer a medián. Čo sa týka polohy, tak Interquartile Range Config1 zapadá do spodnej (lepšej) časti IQR konfigurácie 4, čo sa snaží naznačiť, že Config1 je všeobecne lepšia.

In [ ]:
# TODO: Implementace s vyuzitim knihovni funkce
ALPHA = 0.05
print(test_result:=stats.ttest_ind(configs[0], configs[3], equal_var=False))
print(f"{test_result.pvalue < ALPHA = }")

__OTÁZKA K DOPLNĚNÍ:__

_Jaký je závěr statistického testu?_</br>
Experiment potvrdil, že s vernosťou 95% možno nulovú hypotézu zamietnuť (alpha = 0.05, p-value < alpha). Konfigurácia 1 je rýchlejšia ako 4, lebo z min, max, IQR a mediánu vidno, že konfigurácia 1 nemôže dosahovať horšie výsledky ako konfigurácia 4.

In [ ]:
# TODO vlastni implementace zvoleneho testu
def welchs_test(A, B):
    # Welch's test impl
    len_A, len_B = len(A), len(B)
    var_A, var_B = np.var(A, ddof=1), np.var(B, ddof=1)
    
    degrees_of_freedom = ((var_A / len_A + var_B / len_B)**2) / ((var_A**2 / (len_A**2 * (len_A - 1))) + (var_B**2 / (len_B**2 * (len_B - 1))))
    t_stat = (A.mean() - B.mean()) / np.sqrt((var_A / len_A) + (var_B / len_B))
    p_value = 2 * (1 - stats.t.cdf(abs(t_stat), degrees_of_freedom))
    
    return t_stat, p_value, degrees_of_freedom

welchs_test(configs[0], configs[3])
